# MP2 Part 1 — Commit retrieval (gelkhoda)
Retrieves all WoC commits for my 10 assigned projects plus GitHub stats.
Run top to bottom (Colab or hydra). Retrieval is cached per project in `cache/`, so a crash can simply be re-run.

In [1]:
!python -m pip install -q -U python-woc pandas matplotlib requests tqdm


In [1]:
NETID = 'gelkhoda'
GH_TOKEN = ''   # optional GitHub personal access token (avoids the 60 req/hr limit)
WOC_KEY = ''    # optional WoC API key

import os, time, re, requests
import pandas as pd
from tqdm import tqdm
from woc.remote import WocMapsRemote

woc = WocMapsRemote(base_url='https://worldofcode.org/api/', **({'api_key': WOC_KEY} if WOC_KEY else {}))

assign = pd.read_csv('net2prj.csv')
assign.columns = [c.strip() for c in assign.columns]
mine = assign[assign['netID'].str.strip() == NETID].reset_index(drop=True)
mine['GH'] = mine['GH'].str.strip()
mine

,netID,WoC,GH
0,gelkhoda,gsi-cs-co_chart-fx,https://github.com/GSI-CS-CO/chart-fx
1,gelkhoda,bsaul_inferference,https://github.com/bsaul/inferference
2,gelkhoda,daohu527_dig-into-apollo,https://github.com/daohu527/Dig-into-Apollo
3,gelkhoda,martin2250_opencncpilot,https://github.com/martin2250/OpenCNCPilot
4,gelkhoda,usgs-astrogeology_isis3,https://github.com/USGS-Astrogeology/ISIS3
5,gelkhoda,sakov_enkf-c,https://github.com/sakov/enkf-c
6,gelkhoda,sfc-aqua_quisp,https://github.com/sfc-aqua/quisp
7,gelkhoda,stan-dev_rstanarm,https://github.com/stan-dev/rstanarm
8,gelkhoda,copasi_copasi-dependencies,https://github.com/copasi/copasi-dependencies
9,gelkhoda,almost-matching-exactly_dame-python-package,https://github.com/almost-matching-exactly/dam...


In [3]:
os.makedirs('cache', exist_ok=True)
CHUNK = 10   # WoC max batch size (README: keep chunks < 50 and wait between requests)

def fetch_batch(shas, tries=6):
    for a in range(tries):
        try:
            res, err = woc.get_values_many('commit.tch', shas)
            return {k: v[0] for k, v in res.items()}, err
        except Exception as e:
            wait = 5 * (a + 1)
            print(f'  batch error ({e}); retry in {wait}s'); time.sleep(wait)
    raise RuntimeError('batch failed repeatedly')

def get_project(wocid):
    cache = f'cache/{wocid}.csv'
    if os.path.exists(cache):
        return pd.read_csv(cache, sep=';', keep_default_na=False, dtype={'commit_sha1': str, 'author': str, 'commit message': str})
    shas = list(dict.fromkeys(woc.get_values('p2c', wocid)))
    print(wocid, '->', len(shas), 'commit sha1s')
    if not shas: raise ValueError(f'WoC returned no commits for {wocid} - check the project id')
    got = {}
    for i in tqdm(range(0, len(shas), CHUNK), desc=wocid):
        res, err = fetch_batch(shas[i:i+CHUNK])
        if err: print('  errors:', err)
        got.update(res); time.sleep(1)
    missing = [s for s in shas if s not in got]
    for i in range(0, len(missing), CHUNK):          # one retry pass for anything missed
        res, _ = fetch_batch(missing[i:i+CHUNK]); got.update(res); time.sleep(1)
    missing = [s for s in shas if s not in got]
    if missing: print(f'  WARNING: {len(missing)} commits could not be retrieved')
    rows = [{'project_wocid': wocid, 'commit_sha1': s, 'author': c[2][0],
             'time': int(c[2][1]), 'commit message': c[4]} for s, c in got.items()]
    d = pd.DataFrame(rows).sort_values('time').reset_index(drop=True)
    d.to_csv(cache, sep=';', index=False)
    return d

frames = [get_project(w) for w in mine['WoC']]
summary = pd.concat(frames, ignore_index=True)
summary.to_csv(f'{NETID}_project_summary.csv', sep=';', index=False)
print(len(summary), 'commits saved to', f'{NETID}_project_summary.csv')

gsi-cs-co_chart-fx -> 3641 commit sha1s


gsi-cs-co_chart-fx:   0%|                              | 0/365 [00:00<?, ?it/s]

gsi-cs-co_chart-fx:   0%|                      | 1/365 [00:01<06:14,  1.03s/it]

gsi-cs-co_chart-fx:   1%|                      | 2/365 [00:02<06:14,  1.03s/it]

gsi-cs-co_chart-fx:   1%|▏                     | 3/365 [00:03<06:14,  1.03s/it]

...

almost-matching-exactly_dame-python-package:  96%|▉| 49/51 [00:50<00:02,  1.03s

almost-matching-exactly_dame-python-package:  98%|▉| 50/51 [00:51<00:01,  1.03s

almost-matching-exactly_dame-python-package: 100%|█| 51/51 [00:52<00:00,  1.03s

almost-matching-exactly_dame-python-package: 100%|█| 51/51 [00:52<00:00,  1.04s

34300 commits saved to gelkhoda_project_summary.csv


## WoC statistics

In [4]:
ts = lambda s: pd.to_datetime(s, unit='s', utc=True).strftime('%Y-%m-%d')
woc_stats = (summary.groupby('project_wocid')
             .agg(ncommits=('commit_sha1', 'nunique'), nauthors=('author', 'nunique'),
                  mintime=('time', 'min'), maxtime=('time', 'max'))
             .reindex(mine['WoC']))
woc_stats['min_date'] = woc_stats['mintime'].map(ts)
woc_stats['max_date'] = woc_stats['maxtime'].map(ts)
woc_stats

,ncommits,nauthors,mintime,maxtime,min_date,max_date
WoC,,,,,,
gsi-cs-co_chart-fx,3633,53,1557246279,1758770120,2019-05-07,2025-09-25
bsaul_inferference,328,16,1404407958,1747234885,2014-07-03,2025-05-14
daohu527_dig-into-apollo,425,7,1554864597,1725447268,2019-04-10,2024-09-04
martin2250_opencncpilot,253,15,1474378401,1759704094,2016-09-20,2025-10-05
usgs-astrogeology_isis3,18318,200,1276037385,1759270719,2010-06-08,2025-09-30
sakov_enkf-c,1772,7,1403133737,1762472241,2014-06-18,2025-11-06
sfc-aqua_quisp,4409,87,1524058198,1761562984,2018-04-18,2025-10-27
stan-dev_rstanarm,4146,64,1373994672,1761779325,2013-07-16,2025-10-29
copasi_copasi-dependencies,515,16,1362398954,1773320863,2013-03-04,2026-03-12


## GitHub statistics (stars, forks, last commit date) + validation

In [5]:
S = requests.Session()
S.headers['Accept'] = 'application/vnd.github+json'
if GH_TOKEN: S.headers['Authorization'] = f'Bearer {GH_TOKEN}'

def gh(path, **params):
    r = S.get('https://api.github.com' + path, params=params)
    if r.status_code == 403 and 'rate limit' in r.text.lower():
        raise RuntimeError('GitHub rate limit hit - set GH_TOKEN and re-run')
    r.raise_for_status(); return r

def last_page(r):   # count items using per_page=1 + Link header
    m = re.search(r'[?&]page=(\d+)>; rel="last"', r.headers.get('Link', ''))
    return int(m.group(1)) if m else len(r.json())

gh_rows, recent = [], []
for _, row in mine.iterrows():
    full = row['GH'].rstrip('/').split('github.com/')[1]
    info = gh(f'/repos/{full}').json()                  # follows renames/transfers
    full = info['full_name']
    c1 = gh(f'/repos/{full}/commits', per_page=1)
    last = c1.json()[0]['commit']['committer']['date'][:10]
    contrib = gh(f'/repos/{full}/contributors', per_page=1, anon=1)
    gh_rows.append({'project_wocid': row['WoC'], 'gh_repo': full,
                    'nstars': info['stargazers_count'], 'nforks': info['forks_count'],
                    'lastGHCommitDate': last, 'gh_ncommits': last_page(c1),
                    'gh_ncontributors': last_page(contrib), 'archived': info.get('archived')})
    for c in gh(f'/repos/{full}/commits', per_page=10).json():   # for RecentThemes (Part 3)
        recent.append({'project_wocid': row['WoC'], 'commit_sha1': c['sha'],
                       'author': f"{c['commit']['author']['name']} <{c['commit']['author']['email']}>",
                       'date': c['commit']['author']['date'], 'commit message': c['commit']['message']})
    time.sleep(1)

gh_stats = pd.DataFrame(gh_rows).set_index('project_wocid')
gh_stats.to_csv(f'{NETID}_gh_stats.csv', sep=';')
pd.DataFrame(recent).to_csv(f'{NETID}_gh_recent_commits.csv', sep=';', index=False)

check = woc_stats[['ncommits', 'nauthors', 'max_date']].join(gh_stats[['gh_ncommits', 'gh_ncontributors', 'lastGHCommitDate']])
check['commit_diff'] = check['gh_ncommits'] - check['ncommits']
check   # WoC stops ~Jan 2025, so GH can legitimately be higher; WoC authors are raw name<email> strings (aliases inflate)

,ncommits,nauthors,max_date,gh_ncommits,gh_ncontributors,lastGHCommitDate,commit_diff
WoC,,,,,,,
gsi-cs-co_chart-fx,3633,53,2025-09-25,1189,34,2026-03-13,-2444
bsaul_inferference,328,16,2025-05-14,317,11,2025-05-14,-11
daohu527_dig-into-apollo,425,7,2024-09-04,409,4,2026-02-03,-16
martin2250_opencncpilot,253,15,2025-10-05,205,6,2025-04-28,-48
usgs-astrogeology_isis3,18318,200,2025-09-30,8532,98,2026-09-24,-9786
sakov_enkf-c,1772,7,2025-11-06,1590,4,2026-09-22,-182
sfc-aqua_quisp,4409,87,2025-10-27,2443,33,2026-03-31,-1966
stan-dev_rstanarm,4146,64,2025-10-29,3390,42,2026-09-21,-756
copasi_copasi-dependencies,515,16,2026-03-12,513,12,2026-09-16,-2


In [6]:
# Paste this output into the text cell below
t = woc_stats.join(gh_stats)
print('| Project | #commits | #authors | min time | max time | stars | forks | last GH commit |')
print('|---|---|---|---|---|---|---|---|')
for p, r in t.iterrows():
    print(f"| {p} | {r.ncommits} | {r.nauthors} | {r.mintime} ({r.min_date}) | {r.maxtime} ({r.max_date}) | {r.nstars} | {r.nforks} | {r.lastGHCommitDate} |")

| Project | #commits | #authors | min time | max time | stars | forks | last GH commit |
|---|---|---|---|---|---|---|---|
| gsi-cs-co_chart-fx | 3633 | 53 | 1557246279 (2019-05-07) | 1758770120 (2025-09-25) | 613 | 110 | 2026-03-13 |
| bsaul_inferference | 328 | 16 | 1404407958 (2014-07-03) | 1747234885 (2025-05-14) | 6 | 2 | 2025-05-14 |
| daohu527_dig-into-apollo | 425 | 7 | 1554864597 (2019-04-10) | 1725447268 (2024-09-04) | 2457 | 714 | 2026-02-03 |
| martin2250_opencncpilot | 253 | 15 | 1474378401 (2016-09-20) | 1759704094 (2025-10-05) | 427 | 124 | 2025-04-28 |
| usgs-astrogeology_isis3 | 18318 | 200 | 1276037385 (2010-06-08) | 1759270719 (2025-09-30) | 245 | 182 | 2026-09-24 |
| sakov_enkf-c | 1772 | 7 | 1403133737 (2014-06-18) | 1762472241 (2025-11-06) | 47 | 25 | 2026-09-22 |
| sfc-aqua_quisp | 4409 | 87 | 1524058198 (2018-04-18) | 1761562984 (2025-10-27) | 109 | 43 | 2026-03-31 |
| stan-dev_rstanarm | 4146 | 64 | 1373994672 (2013-07-16) | 1761779325 (2025-10-29) | 401 | 136 

## Results (WoC + GitHub)

| Project | #commits | #authors | min time | max time | stars | forks | last GH commit |
|---|---|---|---|---|---|---|---|
| gsi-cs-co_chart-fx | 3633 | 53 | 1557246279 (2019-05-07) | 1758770120 (2025-09-25) | 613 | 110 | 2026-03-13 |
| bsaul_inferference | 328 | 16 | 1404407958 (2014-07-03) | 1747234885 (2025-05-14) | 6 | 2 | 2025-05-14 |
| daohu527_dig-into-apollo | 425 | 7 | 1554864597 (2019-04-10) | 1725447268 (2024-09-04) | 2457 | 714 | 2026-02-03 |
| martin2250_opencncpilot | 253 | 15 | 1474378401 (2016-09-20) | 1759704094 (2025-10-05) | 427 | 124 | 2025-04-28 |
| usgs-astrogeology_isis3 | 18318 | 200 | 1276037385 (2010-06-08) | 1759270719 (2025-09-30) | 245 | 182 | 2026-09-24 |
| sakov_enkf-c | 1772 | 7 | 1403133737 (2014-06-18) | 1762472241 (2025-11-06) | 47 | 25 | 2026-09-22 |
| sfc-aqua_quisp | 4409 | 87 | 1524058198 (2018-04-18) | 1761562984 (2025-10-27) | 109 | 43 | 2026-03-31 |
| stan-dev_rstanarm | 4146 | 64 | 1373994672 (2013-07-16) | 1761779325 (2025-10-29) | 401 | 136 | 2026-09-21 |
| copasi_copasi-dependencies | 515 | 16 | 1362398954 (2013-03-04) | 1773320863 (2026-03-12) | 7 | 3 | 2026-09-16 |
| almost-matching-exactly_dame-python-package | 501 | 23 | 1564697002 (2019-08-01) | 1758126548 (2025-09-17) | 64 | 15 | 2025-07-26 |

**Notes:**
- WoC counts exceed GitHub's for every project. GitHub's count covers only the default branch, while WoC includes commits from all branches and forks linked to the project; the gap is largest for ISIS3 (+9,786), chart-fx (+2,444) and quisp (+1,966).
- WoC data runs to late 2025–early 2026, so GitHub shows newer commits for most projects.
- OpenCNCPilot and DAME have later WoC commits than their last default-branch commit on GitHub, likely from other branches or forks.
- WoC authors are raw `name <email>` identities, so one person using several emails is counted more than once. This is why WoC author counts exceed GitHub contributor counts.
- 33 commits (<0.1%) could not be retrieved from WoC.
- Three repos have moved: chart-fx → fair-acc/chart-fx, ISIS3 → DOI-USGS/ISIS3, DAME → DAME-FLAME-Python-Package.



# Commit this notebook AND the csv files to your fork!